# 10 — Main GA Loop

**Maps to:** `report/Chapters/Task2.tex` §`T2:GADesign`.
**Ticket:** TICKET-10.

Wires selection, crossover, and mutation into a generational GA loop
with elitism. All operator dependencies are inlined below — scaffolded
sections are marked for replacement once their respective tickets land.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
import tsplib95

PALETTE = "Set2"
%matplotlib inline
sns.set(style='white', context='notebook', palette=PALETTE, rc={'figure.figsize':(14,10)})


---
## Dependencies

Each subsection below provides the operator functions that `run_ga` calls.
Sections marked **SCAFFOLD** contain temporary implementations that must be
replaced with the final versions from their respective ticket notebooks.

### Benchmark Loader (TICKET-04)

> Copied from `notebooks/04_instance_loader.ipynb`.

In [2]:
def load_tsp(path):
    problem = tsplib95.load(str(path))
    nodes = list(problem.get_nodes())
    coords = np.array(
        [problem.node_coords[node] for node in nodes],
        dtype=np.float64,
    )
    diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    dist_matrix = np.sqrt((diff**2).sum(axis=-1))
    return coords, dist_matrix

### SCAFFOLD: Fitness Evaluation (TICKET-05)

> **Temporary implementation.** Replace with the version from
> `notebooks/05_representation_and_fitness.ipynb` when TICKET-05 is completed.

In [3]:
# ── SCAFFOLD START (TICKET-05) ─────────────────────────────────

def tour_length(tour, dist_matrix):
    return dist_matrix[tour, np.roll(tour, -1)].sum()

def is_valid_permutation(chromosome, n_cities):
    return set(chromosome) == set(range(n_cities))

# ── SCAFFOLD END (TICKET-05) ───────────────────────────────────

### SCAFFOLD: Population Initialization (TICKET-06)

> **Temporary implementation.** Replace with the version from
> `notebooks/06_population_init.ipynb` when TICKET-06 is completed.

In [4]:
# ── SCAFFOLD START (TICKET-06) ─────────────────────────────────

def random_population(pop_size, n_cities, rng):
    return np.array([rng.permutation(n_cities) for _ in range(pop_size)])

# ── SCAFFOLD END (TICKET-06) ───────────────────────────────────

### SCAFFOLD: Selection Operators (TICKET-07)

> **Temporary implementation.** Replace with the version from
> `notebooks/07_selection.ipynb` when TICKET-07 is completed.

In [5]:
# ── SCAFFOLD START (TICKET-07) ─────────────────────────────────

def tournament_select(population, fitnesses, k, rng):
    candidates = rng.choice(len(population), size=k, replace=False)
    winner = candidates[np.argmin(fitnesses[candidates])]
    return population[winner].copy()

def roulette_select(population, fitnesses, rng):
    inv = 1.0 / fitnesses
    probs = inv / inv.sum()
    idx = rng.choice(len(population), p=probs)
    return population[idx].copy()

# ── SCAFFOLD END (TICKET-07) ───────────────────────────────────

### Crossover Operators (TICKET-08)

> Copied from `notebooks/08_crossover.ipynb`.

In [6]:
def pmx(parent_a, parent_b, pt1, pt2):
    n = len(parent_a)
    child = np.full(n, -1, dtype=parent_a.dtype)
    child[pt1:pt2 + 1] = parent_a[pt1:pt2 + 1]
    segment_vals = set(child[pt1:pt2 + 1])
    for i in range(pt1, pt2 + 1):
        val = parent_b[i]
        if val not in segment_vals:
            j = i
            while pt1 <= j <= pt2:
                j = np.where(parent_b == parent_a[j])[0][0]
            child[j] = val
    empty = child == -1
    child[empty] = parent_b[empty]
    return child

def ox(parent_a, parent_b, pt1, pt2):
    n = len(parent_a)
    child = np.full(n, -1, dtype=parent_a.dtype)
    child[pt1:pt2 + 1] = parent_a[pt1:pt2 + 1]
    segment_vals = set(child[pt1:pt2 + 1])
    remaining = []
    for i in range(n):
        idx = (pt2 + 1 + i) % n
        if parent_b[idx] not in segment_vals:
            remaining.append(parent_b[idx])
    j = 0
    for i in range(n):
        idx = (pt2 + 1 + i) % n
        if child[idx] == -1:
            child[idx] = remaining[j]
            j += 1
    return child

def cx(parent_a, parent_b):
    n = len(parent_a)
    child = np.full(n, -1, dtype=parent_a.dtype)
    visited = np.zeros(n, dtype=bool)
    pos_in_a = {val: idx for idx, val in enumerate(parent_a)}
    cycle_num = 0
    for start in range(n):
        if visited[start]:
            continue
        cycle_indices = []
        i = start
        while not visited[i]:
            visited[i] = True
            cycle_indices.append(i)
            i = pos_in_a[parent_b[i]]
        source = parent_a if cycle_num % 2 == 0 else parent_b
        for idx in cycle_indices:
            child[idx] = source[idx]
        cycle_num += 1
    return child, cycle_num

### SCAFFOLD: Mutation Operators (TICKET-09)

> **Temporary implementation.** Replace with the version from
> `notebooks/09_mutation.ipynb` when TICKET-09 is completed.

In [7]:
# ── SCAFFOLD START (TICKET-09) ─────────────────────────────────

def swap_mutation(tour, rate, rng):
    tour = tour.copy()
    if rng.random() < rate:
        i, j = rng.choice(len(tour), size=2, replace=False)
        tour[i], tour[j] = tour[j], tour[i]
    return tour

def inversion_mutation(tour, rate, rng):
    tour = tour.copy()
    if rng.random() < rate:
        i, j = sorted(rng.choice(len(tour), size=2, replace=False))
        tour[i:j + 1] = tour[i:j + 1][::-1]
    return tour

# ── SCAFFOLD END (TICKET-09) ───────────────────────────────────

### Diversity Metrics (TICKET-14)

> Copied from `notebooks/14_diversity_metrics.ipynb`.

`diversity` is normalised pairwise Hamming distance for compatibility with downstream experiment notebooks. `edge_overlap` and `edge_diversity` provide a TSP-specific view of shared tour structure.


In [ ]:
def _as_population(population):
    pop = np.asarray(population)
    if pop.ndim != 2:
        raise ValueError("population must be a 2D array-like object")
    if pop.shape[1] == 0:
        raise ValueError("chromosomes must contain at least one city")
    return pop


def pairwise_hamming(population):
    """Return mean pairwise Hamming distance, normalised to [0, 1]."""
    pop = _as_population(population)
    n_individuals, n_genes = pop.shape

    if n_individuals < 2:
        return 0.0

    total = 0.0
    n_pairs = 0
    for i in range(n_individuals - 1):
        total += np.count_nonzero(pop[i] != pop[i + 1:], axis=1).sum()
        n_pairs += n_individuals - i - 1

    return float(total / (n_pairs * n_genes))


def tour_edges(tour):
    """Return the undirected edge set of a cyclic TSP tour."""
    tour = np.asarray(tour)
    return frozenset(
        tuple(sorted((int(tour[i]), int(tour[(i + 1) % len(tour)]))))
        for i in range(len(tour))
    )


def edge_overlap(population):
    """Return mean pairwise fraction of shared undirected tour edges."""
    pop = _as_population(population)
    n_individuals, n_genes = pop.shape

    if n_individuals < 2:
        return 1.0

    edge_sets = [tour_edges(tour) for tour in pop]
    total_overlap = 0.0
    n_pairs = 0

    for i in range(n_individuals - 1):
        for j in range(i + 1, n_individuals):
            total_overlap += len(edge_sets[i] & edge_sets[j]) / n_genes
            n_pairs += 1

    return float(total_overlap / n_pairs)


def edge_difference(population):
    """Return mean pairwise edge-difference distance, normalised to [0, 1]."""
    return float(1.0 - edge_overlap(population))


---
## GA Configuration

In [8]:
@dataclass
class GAConfig:
    pop_size: int = 50
    n_generations: int = 100
    crossover_rate: float = 0.8
    mutation_rate: float = 0.1
    tournament_k: int = 3
    elitism_count: int = 1
    selection_method: str = "tournament"
    crossover_method: str = "pmx"
    mutation_method: str = "swap"
    seed: int = 42
    instrument_diversity: bool = True

## GA Loop Implementation

The loop always logs best fitness and mean fitness at every generation, including the final population after the last update. Diversity instrumentation is controlled by `GAConfig.instrument_diversity`; when enabled, the log also includes Hamming diversity, edge overlap, and edge diversity.


In [ ]:
def run_ga(config, dist_matrix):
    n_cities = dist_matrix.shape[0]
    rng = np.random.default_rng(config.seed)

    xover = {"pmx": pmx, "ox": ox, "cx": cx}[config.crossover_method]
    mutate = {"swap": swap_mutation, "inversion": inversion_mutation}[config.mutation_method]

    population = random_population(config.pop_size, n_cities, rng)
    fitnesses = np.array([tour_length(ind, dist_matrix) for ind in population])

    history = {
        "generation": [],
        "best": [],
        "mean": [],
    }
    if config.instrument_diversity:
        history.update({
            "diversity": [],
            "edge_overlap": [],
            "edge_diversity": [],
        })
    records = []

    def log_generation(gen, population, fitnesses):
        row = {
            "generation": int(gen),
            "best_fitness": float(fitnesses.min()),
            "mean_fitness": float(fitnesses.mean()),
        }
        if config.instrument_diversity:
            overlap = edge_overlap(population)
            row.update({
                "diversity": pairwise_hamming(population),
                "edge_overlap": overlap,
                "edge_diversity": 1.0 - overlap,
            })
        records.append(row)
        history["generation"].append(row["generation"])
        history["best"].append(row["best_fitness"])
        history["mean"].append(row["mean_fitness"])
        if config.instrument_diversity:
            history["diversity"].append(row["diversity"])
            history["edge_overlap"].append(row["edge_overlap"])
            history["edge_diversity"].append(row["edge_diversity"])

    for gen in range(config.n_generations):
        log_generation(gen, population, fitnesses)

        new_pop = []

        elite_idx = np.argsort(fitnesses)[:config.elitism_count]
        for idx in elite_idx:
            new_pop.append(population[idx].copy())

        while len(new_pop) < config.pop_size:
            if config.selection_method == "tournament":
                p1 = tournament_select(population, fitnesses, config.tournament_k, rng)
                p2 = tournament_select(population, fitnesses, config.tournament_k, rng)
            else:
                p1 = roulette_select(population, fitnesses, rng)
                p2 = roulette_select(population, fitnesses, rng)

            if rng.random() < config.crossover_rate:
                if config.crossover_method == "cx":
                    child, _ = xover(p1, p2)
                else:
                    pts = sorted(rng.choice(n_cities, size=2, replace=False))
                    child = xover(p1, p2, pts[0], pts[1])
            else:
                child = p1.copy()

            child = mutate(child, config.mutation_rate, rng)
            new_pop.append(child)

        population = np.array(new_pop[:config.pop_size])
        fitnesses = np.array([tour_length(ind, dist_matrix) for ind in population])

    log_generation(config.n_generations, population, fitnesses)

    best_idx = np.argmin(fitnesses)
    return {
        "best_tour": population[best_idx],
        "best_fitness": float(fitnesses[best_idx]),
        "history": history,
        "log": pd.DataFrame(records),
    }


---
## Smoke Test

Quick end-to-end run with a small population and few generations.
The goal is to verify wiring — not solution quality.

In [10]:
coords, dist_matrix = load_tsp("../data/TSP-dataset/kroA100.tsp")

config = GAConfig(
    pop_size=30,
    n_generations=50,
    crossover_rate=0.8,
    mutation_rate=0.2,
    tournament_k=3,
    elitism_count=1,
    selection_method="tournament",
    crossover_method="pmx",
    mutation_method="swap",
    seed=42,
    instrument_diversity=True,
)

result = run_ga(config, dist_matrix)

In [ ]:
print(f"Best fitness        : {result['best_fitness']:.2f}")
print(f"Known optimal       : 21282")
print(f"Gap                 : {(result['best_fitness'] - 21282) / 21282 * 100:.1f}%")
print(f"Generations logged  : {len(result['log'])}")
if config.instrument_diversity:
    print(f"Final diversity     : {result['history']['diversity'][-1]:.3f}")
    print(f"Final edge diversity: {result['history']['edge_diversity'][-1]:.3f}")
print(f"Valid tour          : {is_valid_permutation(result['best_tour'], 100)}")

base_cols = {"generation", "best_fitness", "mean_fitness"}
metric_cols = {"diversity", "edge_overlap", "edge_diversity"}
assert base_cols.issubset(result["log"].columns)
if config.instrument_diversity:
    assert metric_cols.issubset(result["log"].columns)
else:
    assert metric_cols.isdisjoint(result["log"].columns)
assert len(result["log"]) == config.n_generations + 1
assert result["history"]["best"][-1] <= result["history"]["best"][0],     "Best fitness should not worsen (elitism)"
if config.instrument_diversity:
    assert result["log"]["diversity"].between(0, 1).all()
    assert result["log"]["edge_overlap"].between(0, 1).all()
    assert result["log"]["edge_diversity"].between(0, 1).all()

disabled_config = GAConfig(
    pop_size=10,
    n_generations=2,
    crossover_rate=0.8,
    mutation_rate=0.2,
    tournament_k=3,
    elitism_count=1,
    selection_method="tournament",
    crossover_method="pmx",
    mutation_method="swap",
    seed=7,
    instrument_diversity=False,
)
disabled_result = run_ga(disabled_config, dist_matrix)
assert base_cols.issubset(disabled_result["log"].columns)
assert metric_cols.isdisjoint(disabled_result["log"].columns)
assert "diversity" not in disabled_result["history"]
print()
print("Smoke test passed: fitness logging works, and diversity instrumentation can be toggled.")


### Convergence Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gens = result["history"]["generation"]
axes[0].plot(gens, result["history"]["best"], label="Best", linewidth=2)
axes[0].plot(gens, result["history"]["mean"], label="Mean", linewidth=1, alpha=0.7)
axes[0].set_xlabel("Generation")
axes[0].set_ylabel("Tour Length")
axes[0].set_title("GA Convergence — Smoke Test")
axes[0].legend()

if config.instrument_diversity:
    axes[1].plot(gens, result["history"]["diversity"], label="Hamming diversity", linewidth=2)
    axes[1].plot(gens, result["history"]["edge_diversity"], label="Edge diversity", linewidth=2)
    axes[1].set_xlabel("Generation")
    axes[1].set_ylabel("Diversity")
    axes[1].set_title("Population Diversity per Generation")
    axes[1].set_ylim(0, 1)
    axes[1].legend()
else:
    axes[1].axis("off")
    axes[1].set_title("Diversity instrumentation disabled")

plt.tight_layout()
plt.show()
